# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
from collections import Counter
from sklearn.metrics import classification_report

In [2]:
def majority_vote(row):
    counts = Counter(row)
    return counts.most_common(1)[0][0]

# Import Labels

In [3]:
folder_path = os.getcwd().replace('notebook' , 'dataset')
print(folder_path)

/Users/pranitdas/Desktop/Qillion/dataset


In [4]:
label_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

q_df = pd.read_csv(folder_path + '/dataset4.csv')
queries = q_df['question']
q_df['label'] = q_df['label'].str.lower()
q_df['label'] = q_df['label'].replace(mapping)
label = q_df['label'].str.lower().map(label_mapper)
print(q_df['label'].value_counts())

label
synthesis        29
knowledge        22
evaluation       21
comprehension    20
analysis         19
application      15
Name: count, dtype: int64


In [5]:
label_df = pd.DataFrame()

for filename in ['context_cot', 'context', 'no_context_cot', 'no_context']:
    df = pd.read_csv(folder_path + '/' + filename + '.csv')
    label_df = pd.concat([label_df, df], axis=1)

human_gpt_df = pd.read_csv(folder_path + '/persona_gpt.csv')
cot_human_gpt_df = pd.read_csv(folder_path + '/cot_persona_gpt.csv')

human_llama_df = pd.read_csv(folder_path + '/persona_llama.csv')
cot_human_llama_df = pd.read_csv(folder_path + '/cot_persona_llama.csv')

# Ensemble

In [6]:
context_col = ['cot_context_gpt',	'cot_context_llama',	'zero_shot_context_gpt',	'zero_shot_context_llama',	'few_shot_context_gpt',	'few_shot_context_llama']
no_context_col = ['cot_no_context_gpt',	'cot_no_context_llama',	'zero_shot_no_context_gpt',	'zero_shot_no_context_llama',	'instruct_prompt_no_context_gpt', 'instruct_prompt_no_context_llama']
llama_col = ['cot_context_llama',	'zero_shot_context_llama',	'few_shot_context_llama',	'cot_no_context_llama',	'zero_shot_no_context_llama', 'instruct_prompt_no_context_llama']
gpt_col = ['cot_context_gpt', 'zero_shot_context_gpt', 'few_shot_context_gpt',	'cot_no_context_gpt',	'zero_shot_no_context_gpt', 'instruct_prompt_no_context_gpt']
context_llama_col = ['cot_context_llama',	'zero_shot_context_llama',	'few_shot_context_llama']
context_gpt_col = ['cot_context_gpt', 'zero_shot_context_gpt', 'few_shot_context_gpt']
no_context_llama_col = ['cot_no_context_llama',	'zero_shot_no_context_llama', 'instruct_prompt_no_context_llama']
no_context_gpt_col = ['cot_no_context_gpt',	'zero_shot_no_context_gpt', 'instruct_prompt_no_context_gpt']


In [7]:
final_labels = label_df.apply(majority_vote, axis=1)
context_final_labels = label_df[context_col].apply(majority_vote, axis=1)
no_context_final_labels = label_df[no_context_col].apply(majority_vote, axis=1)
llama_final_labels = label_df[llama_col].apply(majority_vote, axis=1)
gpt_final_labels = label_df[gpt_col].apply(majority_vote, axis=1)
context_llama_final_labels = label_df[context_llama_col].apply(majority_vote, axis=1)
context_gpt_final_labels = label_df[context_gpt_col].apply(majority_vote, axis=1)
no_context_llama_final_labels = label_df[no_context_llama_col].apply(majority_vote, axis=1)
no_context_gpt_final_labels = label_df[no_context_gpt_col].apply(majority_vote, axis=1)

human_gpt_labels = human_gpt_df.apply(majority_vote, axis=1)
cot_human_gpt_labels = cot_human_gpt_df.apply(majority_vote, axis=1)
human_llama_labels = human_llama_df.apply(majority_vote, axis=1)
cot_human_llama_labels = cot_human_llama_df.apply(majority_vote, axis=1)


# Results_technique

In [41]:
print(classification_report(label , [label_mapper[key.lower()] for key in final_labels]))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91        22
           1       0.87      0.65      0.74        20
           2       0.46      0.40      0.43        15
           3       0.81      0.68      0.74        19
           4       0.67      0.90      0.76        29
           5       0.95      0.86      0.90        21

    accuracy                           0.77       126
   macro avg       0.77      0.74      0.75       126
weighted avg       0.78      0.77      0.77       126



In [44]:
print(classification_report(label , [label_mapper[key.lower()] for key in context_final_labels]))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91        22
           1       0.87      0.65      0.74        20
           2       0.45      0.33      0.38        15
           3       0.80      0.63      0.71        19
           4       0.63      0.90      0.74        29
           5       0.90      0.86      0.88        21

    accuracy                           0.75       126
   macro avg       0.76      0.72      0.73       126
weighted avg       0.76      0.75      0.75       126



In [45]:
print(classification_report(label , [label_mapper[key.lower()] for key in no_context_final_labels]))

              precision    recall  f1-score   support

           0       0.87      0.91      0.89        22
           1       0.76      0.65      0.70        20
           2       0.50      0.47      0.48        15
           3       0.81      0.68      0.74        19
           4       0.70      0.90      0.79        29
           5       0.95      0.86      0.90        21

    accuracy                           0.77       126
   macro avg       0.77      0.74      0.75       126
weighted avg       0.77      0.77      0.77       126



In [49]:
print(classification_report(label , [label_mapper[key.lower()] for key in llama_final_labels]))

              precision    recall  f1-score   support

           0       0.87      0.91      0.89        22
           1       0.82      0.70      0.76        20
           2       0.42      0.33      0.37        15
           3       0.81      0.68      0.74        19
           4       0.67      0.90      0.76        29
           5       0.89      0.81      0.85        21

    accuracy                           0.75       126
   macro avg       0.75      0.72      0.73       126
weighted avg       0.76      0.75      0.75       126



In [50]:
print(classification_report(label , [label_mapper[key.lower()] for key in gpt_final_labels]))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91        22
           1       0.81      0.65      0.72        20
           2       0.46      0.40      0.43        15
           3       0.80      0.63      0.71        19
           4       0.68      0.90      0.78        29
           5       0.90      0.86      0.88        21

    accuracy                           0.76       126
   macro avg       0.76      0.73      0.74       126
weighted avg       0.76      0.76      0.76       126



In [8]:
print(classification_report(label , [label_mapper[key.lower()] for key in context_gpt_final_labels]))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91        22
           1       0.87      0.65      0.74        20
           2       0.45      0.33      0.38        15
           3       0.80      0.63      0.71        19
           4       0.63      0.90      0.74        29
           5       0.90      0.86      0.88        21

    accuracy                           0.75       126
   macro avg       0.76      0.72      0.73       126
weighted avg       0.76      0.75      0.75       126



In [9]:
print(classification_report(label , [label_mapper[key.lower()] for key in context_llama_final_labels]))

              precision    recall  f1-score   support

           0       0.84      0.95      0.89        22
           1       0.81      0.65      0.72        20
           2       0.33      0.20      0.25        15
           3       0.81      0.68      0.74        19
           4       0.65      0.90      0.75        29
           5       0.90      0.86      0.88        21

    accuracy                           0.75       126
   macro avg       0.72      0.71      0.71       126
weighted avg       0.74      0.75      0.73       126



In [10]:
print(classification_report(label , [label_mapper[key.lower()] for key in no_context_gpt_final_labels]))

              precision    recall  f1-score   support

           0       0.83      0.91      0.87        22
           1       0.70      0.70      0.70        20
           2       0.54      0.47      0.50        15
           3       0.87      0.68      0.76        19
           4       0.73      0.93      0.82        29
           5       1.00      0.81      0.89        21

    accuracy                           0.78       126
   macro avg       0.78      0.75      0.76       126
weighted avg       0.79      0.78      0.78       126



In [11]:
print(classification_report(label , [label_mapper[key.lower()] for key in no_context_llama_final_labels]))

              precision    recall  f1-score   support

           0       0.91      0.91      0.91        22
           1       0.78      0.70      0.74        20
           2       0.38      0.33      0.36        15
           3       0.72      0.68      0.70        19
           4       0.70      0.90      0.79        29
           5       0.89      0.76      0.82        21

    accuracy                           0.75       126
   macro avg       0.73      0.71      0.72       126
weighted avg       0.75      0.75      0.74       126



# Result_human

In [16]:
print(classification_report(label , [label_mapper[key.lower()] for key in human_gpt_labels]))

              precision    recall  f1-score   support

           0       0.81      0.95      0.88        22
           1       0.74      0.70      0.72        20
           2       0.43      0.40      0.41        15
           3       0.85      0.58      0.69        19
           4       0.66      0.86      0.75        29
           5       1.00      0.76      0.86        21

    accuracy                           0.74       126
   macro avg       0.75      0.71      0.72       126
weighted avg       0.75      0.74      0.74       126



In [17]:
print(classification_report(label , [label_mapper[key.lower()] for key in cot_human_gpt_labels]))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91        22
           1       0.78      0.70      0.74        20
           2       0.46      0.40      0.43        15
           3       0.85      0.58      0.69        19
           4       0.68      0.90      0.78        29
           5       0.90      0.86      0.88        21

    accuracy                           0.76       126
   macro avg       0.76      0.73      0.74       126
weighted avg       0.77      0.76      0.76       126



In [18]:
print(classification_report(label , [label_mapper[key.lower()] for key in human_llama_labels]))

              precision    recall  f1-score   support

           0       0.84      0.95      0.89        22
           1       0.76      0.80      0.78        20
           2       0.36      0.27      0.31        15
           3       0.93      0.74      0.82        19
           4       0.68      0.90      0.78        29
           5       0.94      0.71      0.81        21

    accuracy                           0.76       126
   macro avg       0.75      0.73      0.73       126
weighted avg       0.77      0.76      0.75       126



In [19]:
print(classification_report(label , [label_mapper[key.lower()] for key in cot_human_llama_labels]))

              precision    recall  f1-score   support

           0       0.90      0.86      0.88        22
           1       0.75      0.75      0.75        20
           2       0.54      0.47      0.50        15
           3       0.83      0.79      0.81        19
           4       0.68      0.93      0.78        29
           5       1.00      0.67      0.80        21

    accuracy                           0.77       126
   macro avg       0.78      0.74      0.75       126
weighted avg       0.79      0.77      0.77       126

